# Lab 3: A Model-Selection Scorecard

**Time:** about 75 minutes

Someone will ask the Data & AI team: *"Which open-source LLM should the Fund use?"* The wrong way to answer is to pick
whichever model topped a public leaderboard last week. Leaderboards test general knowledge in English. They say nothing
about the Fund's documents, its members' Swahili, its hardware, or its lawyers' view of a licence.

The right way is a **scorecard**: the Fund's own requirements, weighted by what matters most, with every model tested on
the Fund's own tasks. It's the same approach a procurement committee uses for any tender: agree the criteria and their
weights first, score every bidder against them, then add up. In this lab you'll build one, run it on every model you
have, and produce a shortlist for Day 8.

```
requirements & weights ─┐
                        ├─► scorecard ─► sensitivity check ─► shortlist of 2 for fine-tuning (Day 8)
automatic tests on      │
each local model ───────┤
human ratings (Swahili) ┘
licence & deployment ───┘
```

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **scorecard** | a table scoring each option on several criteria, combined with agreed weights |
| **criterion** | one thing you judge on, e.g. speed or licence |
| **weight** | how much a criterion counts; all the weights add up to 1 |
| **gate** | a requirement that must be met before an option is scored at all |
| **licence** | the legal terms under which a model may be used |
| **grounded answer** | an answer taken from the documents provided, not from the model's memory |
| **refusal** | the model correctly saying the documents don't contain the answer |
| **normalise (a score)** | convert measurements in different units to the same 0-to-1 scale |
| **footprint** | how much memory or disk a model needs |
| **sensitivity check** | testing whether the winner changes if the weights change |
| **shortlist** | the few best options taken forward for more detailed work |

## Where this lab fits

**What you've done so far.** In Lab 1 you ran several open models locally and saw how size, quantisation and language
affect speed and cost. In Lab 2 you wrote and measured prompts for extraction, classification, grounded question
answering and Swahili replies. On Day 2 you compared claims models fairly, with one function that tested every model
the same way.

**What you'll do now.** You'll apply Day 2's fairness to LLM selection. You'll agree the Fund's criteria and weights
first, check licences, run four automatic tests (classification, extraction, grounded Q&A, Swahili) on every model you
have, add your own judgement of Swahili quality, combine everything into a scorecard, check whether the winner survives
different priorities, and write a short recommendation.

**Why it matters.** Choosing a model is a decision the Fund will live with for a long time: it shapes what hardware is
bought, what the legal team signs off, and what members experience. It should rest on evidence from the Fund's own
tasks, set out in a way a committee can follow and challenge.

**What happens next.** The top two models go to Day 8, where you'll fine-tune one on the Fund's material. The scorecard
itself goes into the governance pack on Day 10.

### Theory: multi-criteria decisions

Choosing a model means trading off things that can't be measured in the same units: accuracy, speed, memory, licence
risk, language quality. A **weighted scorecard** is the standard way to do that openly:

1. **Gates first.** Some requirements aren't trade-offs. A model that would send member data outside the Fund is
   excluded, however good it is.
2. **Criteria and weights, agreed before results.** Weights express priorities. Setting them after seeing the scores
   invites (conscious or not) tuning them until a favourite wins.
3. **Put every measurement on the same 0-1 scale**, so a criterion's weight, not its units, decides how much it counts.
4. **Add up** weight × score for each option.
5. **Test the sensitivity**: does the winner change if the priorities change?

The result isn't a mathematical truth. It's a transparent argument: anyone who disagrees can see exactly which weight
or which score they disagree with, and what difference it makes.

## Imports

The same tools as Labs 1 and 2, plus `statistics` (for the median speed) and `numpy`/`matplotlib` for the chart at the
end.

In [ ]:
import json
import os
import re
import statistics
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 170)
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")      # where Ollama is listening

Every model you've downloaded with Ollama is a candidate. Ask Ollama for the list (as in Lab 1), and note each model's
size on disk, which you'll use for the footprint criterion.

In [ ]:
# Every model you've pulled is a candidate. Note each one's size on disk too.
installed = requests.get(OLLAMA_URL + "/api/tags").json()["models"]

CANDIDATES = []
SIZE_GB = {}
for m in installed:
    CANDIDATES.append(m["name"])
    SIZE_GB[m["name"]] = m["size"] / 1e9        # bytes -> gigabytes
CANDIDATES = sorted(CANDIDATES)

print("Candidates:", CANDIDATES)

> **Where we are.** The libraries are loaded, and every model you've pulled is a candidate, with its size on disk noted.
> Before testing any of them, write down what matters.

## 1. Requirements first, models second

Agree the criteria and their weights **before** you see any results. Otherwise it's too tempting to adjust the weights until
your favourite wins. These weights are a starting point for the Fund's first internal assistant. If you'd weight things
differently for your own department, change them now, and write down why.

Each weight is the share of the total score that criterion can earn. `grounded_qa` at 0.25 means a model that answers
every handbook question perfectly earns a quarter of the maximum score from that alone.

In [ ]:
WEIGHTS = {
    "grounded_qa": 0.25,          # answers correctly from the handbook, refuses when the answer isn't there
    "classification": 0.15,      # sorts member messages correctly
    "extraction": 0.15,           # valid, grounded case records from letters
    "swahili": 0.15,              # understands and writes Swahili (automatic test + human rating)
    "speed": 0.10,                # tokens per second on the Fund's hardware
    "footprint": 0.05,            # memory needed: smaller means cheaper servers
    "licence": 0.15,              # can the Fund use it commercially, on-premises, without awkward conditions?
}

# The weights must add up to 1. round() avoids tiny decimal errors like 0.9999999.
print("total weight:", round(sum(WEIGHTS.values()), 6))
pd.Series(WEIGHTS, name="weight")

Data sensitivity isn't a weighted criterion here. It's a **gate**. Every candidate runs on the Fund's own hardware, so
no member data leaves the building. A cloud-only model would be excluded from this scorecard before scoring, however
good it is. (Day 9 compares cloud options separately, for workloads where that trade-off may be acceptable.)

> **Where we are.** The weights are fixed and data sensitivity is a gate. One criterion doesn't need testing at all, only
> reading.

## 2. Licences

"Open-source" doesn't mean "no conditions". Each model comes with a licence that says what you may do with it: some
(Apache 2.0, MIT) allow almost anything; others allow commercial use but add an acceptable-use policy, or extra
conditions for very large organisations. The Fund needs to be able to use a model commercially, on its own servers,
without surprises.

The notes below are a **summary for the lab**, not legal advice. Licences change, so the Fund's legal team must read
the current text before any production use. Each licence gets a score from 1 (unknown or risky) to 5 (no concerns).

A model's name in Ollama is `family:size` (e.g. `qwen2.5:1.5b`), and the licence depends on the family, so
`licence_for` looks up the part before the colon.

In [ ]:
# Model family -> (licence summary, score from 1 to 5 for the Fund's use)
LICENCES = {
    "qwen2.5": ("Apache 2.0 for most sizes; the 3B and 72B versions use Qwen's own licence", 4),
    "llama3.2": ("Llama 3.2 Community Licence: commercial use allowed, with conditions and an acceptable-use policy", 3),
    "llama3.1": ("Llama 3.1 Community Licence: commercial use allowed, with conditions", 3),
    "gemma2": ("Gemma Terms of Use: commercial use allowed, with a prohibited-use policy", 3),
    "mistral": ("Apache 2.0", 5),
    "phi3.5": ("MIT", 5),
    "phi3": ("MIT", 5),
}


def licence_for(model):
    """The licence summary and score for a model, looked up by its family (the part before the colon)."""
    family = model.split(":")[0]           # "qwen2.5:1.5b" -> "qwen2.5"
    if family in LICENCES:
        return LICENCES[family]
    return ("Unknown: check before use", 1)


# One row per candidate.
rows = []
for model in CANDIDATES:
    summary, licence_score = licence_for(model)
    rows.append({"model": model, "licence summary": summary, "score (1-5)": licence_score})
pd.DataFrame(rows)

> **Where we are.** Each model family has a licence summary and a score. The remaining criteria need the models to
> actually do the Fund's tasks.

### Theory: what makes a test set useful for choosing a model

The tests here are small, but they follow the rules that make any comparison fair:

- **The Fund's own material.** Member messages, Fund letters and handbook sections, not general-knowledge quizzes.
- **Fixed and identical for every model.** Same items, same prompts, temperature 0. Any difference comes from the model.
- **Checks that a program can apply**: correct category, valid JSON with nothing invented, required phrases present, the
  refusal sentence when the answer isn't there.
- **Tests of the failure that matters most.** Two of the Q&A questions *can't* be answered from their document, because
  a model that bluffs is more dangerous than one that's sometimes unhelpful.

Small test sets give rough rankings. Ten questions can separate a strong model from a weak one; they can't reliably
separate two close models. The notes at the end say what to do before production.

## 3. The automatic tests

Small, fixed test sets, built from the Fund's own material. The same prompts go to every model at temperature 0, so the
comparison is fair and repeatable: any difference in score comes from the model, not from luck.

This `chat` function takes the model as its first argument (so the same test can run on each candidate) and returns
two things: the reply text and the generation speed in tokens per second. If Ollama doesn't report a duration, the
speed is `np.nan` ("not a number"), which pandas treats as missing.

In [ ]:
def chat(model, prompt, system=None, fmt=None, max_tokens=300):
    """Send one prompt to a model in Ollama. Returns (reply text, generation speed in tokens per second)."""
    messages = []
    if system is not None:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    body = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": 0, "num_predict": max_tokens, "seed": 42},
    }
    if fmt is not None:
        body["format"] = fmt          # force the answer into this JSON schema

    reply = requests.post(OLLAMA_URL + "/api/chat", json=body, timeout=300).json()

    # Speed = tokens generated / seconds spent generating (Ollama reports nanoseconds).
    if reply.get("eval_duration"):
        tokens_per_second = reply["eval_count"] / (reply["eval_duration"] / 1e9)
    else:
        tokens_per_second = np.nan
    return reply["message"]["content"], tokens_per_second

### Test 1: classification (30 member messages)

The same task as Lab 2: put each member message in the right category. Take a balanced set of 30 messages (up to 4 per
category, shuffled with a different seed from Lab 2's), and use a schema so every reply is a real category name.
`DEFS` gives the short meaning of each category on one line.

In [ ]:
fb = pd.read_csv("member_feedback.csv").drop_duplicates(subset=["text"])
CATEGORIES = sorted(fb["category"].unique())

# Up to 4 messages per category, 30 in total.
cls_set = fb.sample(frac=1, random_state=7).groupby("category").head(4).head(30)

CAT_SCHEMA = {
    "type": "object",
    "properties": {"category": {"type": "string", "enum": CATEGORIES}},
    "required": ["category"],
}

DEFS = ("compliment: praise or thanks | contribution_not_posted: contributions missing from statement | customer_service: "
        "queues, staff, calls not answered | delayed_benefit: claim lodged but not paid | digital_services: app, portal, USSD, OTP | "
        "employer_not_remitting: employer deducts but doesn't pay | general_enquiry: question, no complaint | pension_payment: "
        "monthly pension missing or suspended | registration_details: name, NIDA, phone, duplicate member numbers")
print(len(cls_set), "test messages")

The test function sends each message, counts the correct answers, and keeps every call's speed. `try ... except` means
"try this; if one of these errors happens, do the `except` part instead": a reply that isn't valid JSON, or has no
`category`, simply scores 0 rather than stopping the whole test. `pass` means "do nothing".

In [ ]:
def test_classification(model):
    """Share of the 30 messages put in the right category, and the speed of each call."""
    correct = 0
    speeds = []
    for text, true_category in zip(cls_set["text"], cls_set["category"]):
        prompt = "Categories: " + DEFS + "\n\nMessage (English, Swahili or both): <<<" + text + ">>>\nClassify it."
        reply, tokens_per_second = chat(model, prompt, fmt=CAT_SCHEMA, max_tokens=30)
        speeds.append(tokens_per_second)
        try:
            if json.loads(reply)["category"] == true_category:
                correct += 1
        except (json.JSONDecodeError, KeyError):
            pass                            # a broken reply simply scores 0
    return correct / len(cls_set), speeds

### Test 2: extraction (four documents)

A record scores a point only if the JSON (JavaScript Object Notation, a plain-text format for structured data) is valid
**and** it passes the grounding check from Lab 2: no invented member number or name. A smaller schema than Lab 2's keeps
the test quick. Only the letters are used (`letter_*.txt`), not the memo.

In [ ]:
# The letters only (not the memo).
docs = {}
for path in sorted(Path("documents").glob("letter_*.txt")):
    docs[path.name] = path.read_text(encoding="utf-8")

EX_SCHEMA = {
    "type": "object",
    "properties": {
        "writer_name": {"type": "string"},
        "member_no": {"type": ["string", "null"]},
        "summary_in_english": {"type": "string"},
    },
    "required": ["writer_name", "member_no", "summary_in_english"],
}
print(len(docs), "letters")

For each letter: ask for the record; skip it (no point) if the JSON is broken (`continue` jumps to the next letter);
then check the member number and the writer's surname against the letter, as in Lab 2. A missing member number is fine
(the letter may not give one); an invented one isn't. `bool(writer) and ...` first checks that a name was given at all.

In [ ]:
def test_extraction(model):
    """Share of the letters that give valid JSON with no invented member number or name."""
    good = 0
    for text in docs.values():
        prompt = ("Letter:\n<<<\n" + text + "\n>>>\nExtract writer_name, member_no (null if absent) and a two-sentence "
                  "English summary. Use only what the letter says.")
        reply, _ = chat(model, prompt, fmt=EX_SCHEMA, max_tokens=250)

        try:
            record = json.loads(reply)
        except json.JSONDecodeError:
            continue                        # invalid JSON scores 0

        # The member number must be empty, or its digits must appear in the letter.
        member_no = record.get("member_no")
        if member_no in (None, "", "null"):
            member_ok = True
        else:
            member_ok = re.sub(r"\D", "", str(member_no)) in re.sub(r"\D", "", text)

        # The writer's surname must appear in the letter.
        writer = record.get("writer_name")
        name_ok = bool(writer) and writer.split()[-1] in text

        if member_ok and name_ok:
            good += 1
    return good / len(docs)

### Test 3: grounded question answering

This carries the biggest weight, because answering from the Fund's documents (and refusing when they don't contain the
answer) is the first use case. Each question comes with the handbook file to use and the words a correct answer must
contain. The last two questions can't be answered from their document, so the only correct answer is the refusal
sentence; `REFUSAL[:25]` checks for its first 25 characters.

In [ ]:
REFUSAL = "The documents provided don't answer this."
QA_SYSTEM = ("Answer ONLY from the document. If it does not contain the answer, reply exactly: " + REFUSAL + " "
             "Quote figures exactly as written. Keep answers to one or two sentences.")

# (question, document to give the model, words the answer must contain)
QA_SET = [
    ("How many months of contributions are needed for an old age pension?", "handbook/04_old_age_pension.md", ["180"]),
    ("What is the compulsory retirement age?", "handbook/04_old_age_pension.md", ["60"]),
    ("How much is the funeral grant?", "handbook/07_funeral_grant.md", ["500,000"]),
    ("For how many weeks is maternity benefit paid?", "handbook/08_maternity_benefit.md", ["12"]),
    ("What percentage of the average wage does unemployment benefit pay?", "handbook/10_unemployment_benefit.md", ["33.3"]),
    ("Who pays compensation for injuries at work?", "handbook/09_employment_injury.md", ["Workers Compensation Fund"]),
    ("What is the minimum voluntary contribution per month?", "handbook/02_registration.md", ["20,000"]),
    ("By when must a complaint be acknowledged?", "handbook/12_complaints_and_appeals.md", ["2 working days"]),
    # These two can't be answered from the document given: the right answer is the refusal.
    ("What interest rate does the Fund pay on contributions?", "handbook/07_funeral_grant.md", [REFUSAL[:25]]),
    ("How many branches does the Fund have in Dodoma?", "handbook/05_invalidity_benefit.md", [REFUSAL[:25]]),
]

The check is deliberately simple: an answer counts as correct if it contains every required phrase, ignoring upper and
lower case. It can't judge a well-worded answer that writes "one hundred and eighty months" instead of "180", but it is
the same for every model, which is what a fair comparison needs.

In [ ]:
def test_grounded_qa(model):
    """Share of handbook questions answered with the right fact, or correctly refused."""
    correct = 0
    for question, path, must_contain in QA_SET:
        document = Path(path).read_text(encoding="utf-8")
        reply, _ = chat(model, "DOCUMENT:\n" + document + "\n\nQUESTION: " + question, system=QA_SYSTEM, max_tokens=120)

        # Correct only if every required phrase is in the answer (ignoring upper/lower case).
        all_found = True
        for phrase in must_contain:
            if phrase.lower() not in reply.lower():
                all_found = False
        if all_found:
            correct += 1
    return correct / len(QA_SET)

### Test 4: Swahili (automatic part)

Three questions asked in Swahili, with the answer expected in Swahili. The automatic check only looks for the right fact
and for a few common Swahili words. Quality of language is for people to judge (Section 5).

Each question can earn two points: one for containing the right fact, one for using at least two common Swahili words
(a rough test that the answer is in Swahili at all). `re.findall(r"\w+", ...)` splits the reply into words; `&` between
two sets gives the words they share.

In [ ]:
# (question in Swahili, document, the fact the answer must contain)
SW_SET = [
    ("Umri wa lazima wa kustaafu ni miaka mingapi?", "handbook/04_old_age_pension.md", "60"),
    ("Msaada wa mazishi ni kiasi gani?", "handbook/07_funeral_grant.md", "500,000"),
    ("Fao la uzazi linalipwa kwa wiki ngapi?", "handbook/08_maternity_benefit.md", "12"),
]
SW_WORDS = {"ni", "na", "ya", "wa", "kwa", "la", "za", "miaka", "wiki", "shilingi"}


def test_swahili(model):
    """Two points per question: one for the right fact, one for answering in Swahili. Returns (share of points, answers)."""
    points = 0
    answers = []
    for question, path, fact in SW_SET:
        document = Path(path).read_text(encoding="utf-8")
        prompt = "DOCUMENT:\n" + document + "\n\nSWALI: " + question + "\nJibu kwa Kiswahili, kwa sentensi moja."
        reply, _ = chat(model, prompt, max_tokens=100)
        answers.append(reply.strip())

        # Point 1: the fact is there (spaces removed, so "500 000" doesn't match but "500,000" does).
        if fact in reply.replace(" ", ""):
            points += 1

        # Point 2: at least two common Swahili words are used.
        words = set(re.findall(r"\w+", reply.lower()))
        if len(words & SW_WORDS) >= 2:
            points += 1
    return points / (2 * len(SW_SET)), answers

> **Where we are.** Four tests are written as functions: classification, extraction, grounded Q&A and Swahili. Each takes
> a model name and returns a score. Now run them all.

## 4. Run every test on every model

On a laptop GPU (graphics processing unit) this takes a few minutes per model; on a CPU, longer. Each model is loaded in
turn, so only one needs to fit in memory at a time.

For each model the loop:

1. sends a tiny "Hello" first, so the model is loaded into memory before anything is timed (otherwise the first test
   would look slow);
2. runs the four tests;
3. records the results, the **median** speed (the middle value of all the classification calls, which isn't thrown
   off by one unusually slow call), the size and the licence score.

In [ ]:
rows = []
swahili_answers = {}      # model -> its three Swahili answers, for you to read in Section 5

for model in CANDIDATES:
    start = time.perf_counter()
    chat(model, "Hello", max_tokens=5)                      # load the model before timing anything

    classification, speeds = test_classification(model)
    extraction = test_extraction(model)
    grounded_qa = test_grounded_qa(model)
    swahili_auto, swahili_answers[model] = test_swahili(model)
    minutes = (time.perf_counter() - start) / 60

    rows.append({
        "model": model,
        "classification": classification,
        "extraction": extraction,
        "grounded_qa": grounded_qa,
        "swahili_auto": swahili_auto,
        "tokens_per_second": statistics.median(speeds),
        "size_gb": SIZE_GB[model],
        "licence_score": licence_for(model)[1],      # [1] = the score part of (summary, score)
        "minutes": minutes,
    })
    print(f"{model}: done in {minutes:.1f} min")

raw = pd.DataFrame(rows).set_index("model")
raw.round(3)

> **Where we are.** Every model has automatic scores on four tests, a median speed, a size and a licence score, in the
> `raw` table. One thing no automatic test can judge well is still missing.

## 5. The human part: Swahili quality

Automatic checks can tell whether an answer contains "60". They can't tell whether the Swahili is natural, polite and
correct. You can. Read each model's Swahili answers and give each model a score from 1 (unusable) to 5 (would send it to
a member as is).

In [ ]:
for model in CANDIDATES:
    print("===", model)
    for i in range(len(SW_SET)):
        print("  ", SW_SET[i][0])                     # the question
        print("   ->", swahili_answers[model][i])     # this model's answer

Now enter your ratings. The cell starts every model at 3 (average); change each number to your own score. If several
people in the room speak Swahili, agree the scores together, or take the average of everyone's scores.

In [ ]:
# Replace the 3s with your own scores (1-5).
HUMAN_SWAHILI = {}
for model in CANDIDATES:
    HUMAN_SWAHILI[model] = 3
HUMAN_SWAHILI

> **Where we are.** You've read every model's Swahili answers and rated them. All the measurements are in. Now combine
> them.

## 6. The scorecard

The measurements are in different units: accuracy shares, tokens per second, gigabytes, a 1-5 licence score. To combine
them, turn every measurement into a **0-to-1 score**, where 1 is best:

- accuracy-type tests are already 0-1;
- Swahili combines the automatic check (40%) and your human rating (60%). People are the better judges here. The rating
  1-5 becomes 0-1 with `(rating - 1) / 4`, so 1 → 0, 3 → 0.5 and 5 → 1;
- speed is scored relative to the fastest candidate (fastest = 1, half as fast = 0.5);
- footprint is scored relative to the smallest (smallest = 1, twice as big = 0.5), so smaller is better;
- the licence score is divided by 5.

In [ ]:
score = pd.DataFrame(index=raw.index)

score["grounded_qa"] = raw["grounded_qa"]
score["classification"] = raw["classification"]
score["extraction"] = raw["extraction"]

# Human rating 1-5 -> 0-1: (rating - 1) / 4.
human = pd.Series(HUMAN_SWAHILI)
score["swahili"] = 0.4 * raw["swahili_auto"] + 0.6 * (human - 1) / 4

# Fastest model = 1; a model half as fast = 0.5.
score["speed"] = raw["tokens_per_second"] / raw["tokens_per_second"].max()

# Smallest model = 1; a model twice as big = 0.5.
score["footprint"] = raw["size_gb"].min() / raw["size_gb"]

score["licence"] = raw["licence_score"] / 5

score.round(3)

The total is a **weighted sum**: each criterion's 0-1 score times its weight, added up. A perfect model would score 1.
Sort with the best at the top.

In [ ]:
# The total: each criterion's score times its weight, added up.
score["TOTAL"] = 0.0
for criterion, weight in WEIGHTS.items():
    score["TOTAL"] = score["TOTAL"] + score[criterion] * weight

scorecard = score.sort_values("TOTAL", ascending=False)
scorecard.round(3)

A **stacked bar chart** shows where each model earns its points: one bar per model, made of coloured pieces, one per
criterion. `left` keeps track of where the next piece should start, so the pieces sit end to end.
`invert_yaxis()` puts the best model at the top.

In [ ]:
# A stacked bar per model: each coloured piece is one criterion's weighted points.
plt.figure(figsize=(9, 0.6 * len(scorecard) + 1.5))
left = np.zeros(len(scorecard))                         # where the next piece starts

for criterion, weight in WEIGHTS.items():
    points = (scorecard[criterion] * weight).values
    plt.barh(scorecard.index, points, left=left, label=criterion)
    left = left + points

plt.gca().invert_yaxis()                                # best model at the top
plt.xlabel("weighted score")
plt.title("Model scorecard: where each model earns its points", loc="left")
plt.legend(ncol=4, fontsize=8, loc="lower right")
plt.show()

The stacked bars show *why* a model ranks where it does. A model that wins mostly on speed and size is a different choice
from one that wins on grounded answers.

> **Where we are.** The scorecard ranks the models, and the stacked bars show where each earns its points. Before
> declaring a winner, check how much the ranking depends on the weights you chose.

## 7. Would a different set of priorities change the answer?

Weights are judgements, and reasonable people disagree about them. A **sensitivity check** tests whether the winner
survives that disagreement. For each criterion in turn:

1. copy the weights and double that one criterion's weight (`dict(WEIGHTS)` makes a copy, so the original stays as it was);
2. recompute every model's total, and divide by the new sum of the weights so the totals stay on a 0-1 scale;
3. note which model comes top (`idxmax()` gives the name of the row with the largest value).

In [ ]:
rows = []

for doubled in WEIGHTS:
    # A copy of the weights with one criterion doubled.
    new_weights = dict(WEIGHTS)
    new_weights[doubled] = new_weights[doubled] * 2

    # Recompute the totals, divide by the new sum of weights, and find the top model.
    total = 0.0
    for criterion, weight in new_weights.items():
        total = total + score[criterion] * weight
    total = total / sum(new_weights.values())

    rows.append({"criterion doubled": doubled, "winner": total.idxmax()})

pd.DataFrame(rows)

If the same model wins whatever you double, the choice is safe and the argument is over. If the winner flips, then
the decision really rests on that criterion, and that's the question to take to management, stated plainly: "If Swahili
quality matters most, choose A; if speed on existing hardware matters most, choose B."

> **Where we are.** You know whether the winner holds up when each criterion's weight is doubled, and if not, which
> criterion the decision really depends on. Time to write it down.

## 8. Your shortlist

Save the scorecard as a CSV file (it's evidence for the governance pack on Day 10), and take the top two models forward
to Day 8. `scorecard.index[:2]` is the names of the first two rows.

In [ ]:
Path("outputs").mkdir(exist_ok=True)
scorecard.round(4).to_csv("outputs/model_scorecard.csv")

shortlist = list(scorecard.index[:2])
print("Shortlist for Day 8 fine-tuning:", shortlist)

Write a short recommendation, five sentences at most, for the head of ICT and the Director of Operations. Use this
structure:

1. **Recommendation:** which model(s), for which first use case.
2. **Evidence:** the two or three scores that decided it.
3. **Risks:** what the model got wrong in testing, and how the design will contain it (grounding, refusal, human review).
4. **Hardware:** what it needs to run (size, speed on the test machine).
5. **Next step:** fine-tuning and retrieval tests on Days 8 and 9, and legal review of the licence.

**Your turn.** Write it in the cell below. This is the kind of document a governance committee will ask to see on Day 10.

*Your recommendation here.*

---

## Looking back, looking ahead

**What you did in this lab.** You built a model-selection scorecard: a data gate, criteria and weights agreed first,
licence scores, four automatic tests on the Fund's own material, a human rating for Swahili, normalised scores combined
with weights, a sensitivity check, a saved scorecard and a short recommendation.

**What you did today.** Day 7 took you from running an LLM locally (Lab 1), to getting reliable work out of it with
prompts (Lab 2), to choosing which models the Fund should take forward (Lab 3).

**The ideas to carry forward.**

- Choose models on the Fund's tasks, not on public leaderboards.
- Agree criteria and weights before seeing results; apply gates first.
- People judge language quality better than automatic checks.
- Test whether the decision survives different priorities.
- Small models answering from memory get policy wrong; grounding and refusal are design requirements.

**Tomorrow (Day 8).** You'll adapt a model to the Fund: build a clean instruction dataset from the FAQ and handbook,
fine-tune with LoRA, and test honestly whether it helped.

## Notes for going further

- **Test bigger models.** On a server with a 24 GB GPU you can run 7-8B models (e.g. `llama3.1:8b`, `qwen2.5:7b`,
  `mistral:7b`) and quantised 14B models. Add them to Ollama and rerun this notebook unchanged. Expect clearly better
  grounded answers and Swahili, at a quarter of the speed.
- **Grow the test sets.** Ten questions is enough to *rank* models roughly, not to *certify* one. Before production, aim
  for a few hundred test questions written by the people who answer members' questions today.
- **Retest on every model update.** A new version of the same model can behave differently. Keep this notebook and the test
  sets under version control and rerun them before switching.